Obter o arquivo carros-fipe.csv

In [1]:
!wget https://raw.githubusercontent.com/esensato/ssa-2026-01/refs/heads/main/carros-fipe.csv

--2026-03-13 17:42:51--  https://raw.githubusercontent.com/esensato/ssa-2026-01/refs/heads/main/carros-fipe.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5082 (5.0K) [text/plain]
Saving to: ‘carros-fipe.csv’

carros-fipe.csv     100%[===================>]   4.96K  --.-KB/s    in 0s      

2026-03-13 17:42:51 (72.7 MB/s) - ‘carros-fipe.csv’ saved [5082/5082]



Criar o Dataset para o arquivo importado

In [2]:
import pandas as pd
df = pd.read_csv('carros-fipe.csv', delimiter=";")
df.head(10)

,registro_id,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado
0,1,59,5940,2011-3,2025-01-10,"R$ 88.629,00","1.200,00","89.829,00",120000,ABC1D23,Joao Silva,123.456.789-09,SP
1,2,59,5940,2021-3,10/01/2025,88629.00,1200.00,89829.00,120.000,ABC1D23,JOAO SILVA,12345678909,SP
2,3,59,5940,2020-3,01-10-2025,"88,629.00","1.200,00","89.829,00",120000km,ABC1D23,joao alberto silva,123-456-789-10,RJ
3,4,21,4828,2010-5,2025/02/15,"R$45.990,00","900,00","46.890,00",98000,DEF-4K56,Maria Souza Franco,987.654.321-00,SP
4,5,21,4828,2010-5,15/02/2025,45990,900,46890,98.000,DEF4K56,maria augusta souza,98765432100,SP
5,6,21,4828,2010-6,02-15-2025,"R$45990,00","900,00","46890,00",98-000,DEF4K56,MARIA SOUZA,987-654-321-00,XX
6,7,23,1043,2020-1,2025-03-20,"73.500,00","1.000,00","74.500,00",65000,GHI-7L89,Carlos Pereira,741.852.963-11,RJ
7,8,23,1044,2003-1,20/03/2025,73500,1000,74500,65.000,GHI7L89,carlos pereira,74185296311,RJ
8,9,23,6831,2020-1,03-20-2025,"R$ 73.500,00","1000,00","74.400,00",65000km,GHI7L89,CARLOS PEREIRA,741-852-963-11,SP
9,10,22,9120,2019-3,2025-04-05,"52.300,00","850,00","53.150,00",42000,JKL-2P45,Ana Lima,159.753.486-22,SP


Apagar a coluna registro_id

In [3]:
df = df.drop(columns=['registro_id'])

Colocar no padrão a data da venda

In [4]:
def parse_data(valor):
	formatos = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y", "%m-%d-%Y"]
	for f in formatos:
		try:
			return pd.to_datetime(valor, format=f)
		except:
			pass
	return pd.NaT


In [5]:
df['data_venda'] = df['data_venda'].apply(parse_data)

Padronizar valores monetários (valor_venda, taxa_servico e valor_total)

In [6]:
def validar_numero(numero):
  try:
    return float(numero)
  except ValueError:
    tmp = "";
    pos = 0;
    for i in numero:
      if i.isnumeric():
        tmp += i
      if (i == '.' or i == ',') and pos == len(numero) - 3:
        tmp += i
      pos += 1
    return float(tmp.replace(',', '.'))

In [7]:
df['valor_venda'] = df['valor_venda'].apply(validar_numero)
df['taxa_servico'] = df['taxa_servico'].apply(validar_numero)
df['valor_total'] = df['valor_total'].apply(validar_numero)

Validar o campo km_rodados deixando somente números

In [8]:
def validar_km(numero):
  try:
    return int(numero)
  except ValueError:
    tmp = "";
    for i in numero:
      if i.isnumeric():
        tmp += i
    return int(tmp)

In [9]:
df['km_rodados'] = df['km_rodados'].apply(validar_km)

Outros ajustes: cpf, placa e nome do cliente

In [10]:
df['cpf'] = df['cpf'].str.replace('.', '')
df['cpf'] = df['cpf'].str.replace('-', '')
df['placa'] = df['placa'].str.replace('-', '')
df['cliente'] = df['cliente'].str.upper()

Criar um dataframe para conter os itens descartados (antes de descartar propriamente os campos obrigatórios)

In [11]:
descarte = pd.DataFrame(df[df["cpf"].isna()])
df = df.dropna(subset="cpf")

In [12]:
descarte = pd.concat([descarte, df[df["placa"].isna()]], ignore_index=True)
df = df.dropna(subset="placa")

In [13]:
df

,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado
0,59,5940,2011-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP
1,59,5940,2021-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP
2,59,5940,2020-3,2025-10-01,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO ALBERTO SILVA,12345678910,RJ
3,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA FRANCO,98765432100,SP
4,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA AUGUSTA SOUZA,98765432100,SP
5,21,4828,2010-6,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA,98765432100,XX
6,23,1043,2020-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ
7,23,1044,2003-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ
8,23,6831,2020-1,2025-03-20,73500.0,1000.0,74400.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,SP
9,22,9120,2019-3,2025-04-05,52300.0,850.0,53150.0,42000,JKL2P45,ANA LIMA,15975348622,SP


In [14]:
descarte

,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado
0,21,4828,2010-5,2025-07-10,45990.0,900.0,46890.0,98000,STW8V90,MARCOS ALVES,NaN,SP
1,44,XXXX,2017-1,2025-05-11,39900.0,700.0,40600.0,87000,NaN,MARCOS MENDES,32165498733,SP
2,59,5940,2014-1,2025-06-02,88629.0,1200.0,89829.0,120000,NaN,JULIANA COSTA,95135725866,RJ
3,21,4828,2018-2,2025-12-01,45990.0,900.0,46890.0,98000,NaN,BRUNA CARVALHO,78945612355,SP


Criar uma função para obter os dados do veículo na URL https://parallelum.com.br/fipe/api/v1/carros/marcas/marca_id/modelos/modelo_id/anos/ano_modelo, substituindo marca_id, modelo_id e ano_modelo

In [36]:
import requests

def obter_dados_veiculo(marca_id, modelo_id, ano_modelo):
    url = f"https://parallelum.com.br/fipe/api/v1/carros/marcas/{marca_id}/modelos/{modelo_id}/anos/{ano_modelo}"
    response = requests.get(url)
    return {
        "status_code": response.status_code,
        "valor_fipe": response.json().get("Valor"),
        "marca_fipe": response.json().get("Marca")
    }

In [37]:
df[["status_code","valor_fipe","marca_fipe"]] = df.apply(
    lambda row: pd.Series(
        obter_dados_veiculo(row["marca_id"], row["modelo_id"], row["ano_modelo"])
    ),
    axis=1
)

In [38]:
df

,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado,status_code,valor_fipe,marca_fipe
0,59,5940,2011-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,500.0,NaN,NaN
1,59,5940,2021-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,200.0,"R$ 149.786,00",VW - VolksWagen
2,59,5940,2020-3,2025-10-01,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO ALBERTO SILVA,12345678910,RJ,200.0,"R$ 141.217,00",VW - VolksWagen
3,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA FRANCO,98765432100,SP,200.0,"R$ 25.693,00",Fiat
4,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA AUGUSTA SOUZA,98765432100,SP,200.0,"R$ 25.693,00",Fiat
5,21,4828,2010-6,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA,98765432100,XX,500.0,NaN,NaN
6,23,1043,2020-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ,500.0,NaN,NaN
7,23,1044,2003-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ,200.0,"R$ 16.875,00",GM - Chevrolet
8,23,6831,2020-1,2025-03-20,73500.0,1000.0,74400.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,SP,500.0,NaN,NaN
9,22,9120,2019-3,2025-04-05,52300.0,850.0,53150.0,42000,JKL2P45,ANA LIMA,15975348622,SP,500.0,NaN,NaN


Decartar as linhas onde status_code seja igual a 500...

In [40]:
carros_fipe_descarte_endpoint = df[df["status_code"] == 500.0]

In [41]:
carros_fipe_descarte_endpoint

,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado,status_code,valor_fipe,marca_fipe
0,59,5940,2011-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,500.0,NaN,NaN
5,21,4828,2010-6,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA,98765432100,XX,500.0,NaN,NaN
6,23,1043,2020-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ,500.0,NaN,NaN
8,23,6831,2020-1,2025-03-20,73500.0,1000.0,74400.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,SP,500.0,NaN,NaN
9,22,9120,2019-3,2025-04-05,52300.0,850.0,53150.0,42000,JKL2P45,ANA LIMA,15975348622,SP,500.0,NaN,NaN
10,22,9120,2019-3,2025-04-05,52300.0,850.0,53150.0,42000,JKL2P45,ANA LIMA,15975348622,RJ,500.0,NaN,NaN
11,22,9120,2019-3,2025-05-04,52300.0,850.0,53150.0,42000,JKL2P45,ANA LIMA,15975348622,ZZ,500.0,NaN,NaN
12,44,7710,2017-1,2025-05-11,39900.0,700.0,40600.0,87000,MNO9T12,PAULO MENDES,32165498733,SP,500.0,NaN,NaN
14,44,7710,2017-1,2025-11-05,39900.0,700.0,40600.0,87000,MNO9T12,PAULO MENDES,32165498733,RJ,500.0,NaN,NaN
15,21,5940,2014-1,2025-01-10,88629.0,1200.0,89829.0,120000,XYZ1A11,LUCAS PRADO,45612378944,SP,500.0,NaN,NaN


Agora, remover o que estiver incorreto do ds (status_code = 500) e manter status_code = 200

In [42]:
df = df[df["status_code"] == 200]

Converter o valor_fipe para número

In [44]:
df['valor_fipe'] = df['valor_fipe'].apply(validar_numero)

/tmp/ipykernel_230/2610974777.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valor_fipe'] = df['valor_fipe'].apply(validar_numero)


Calcular o lucro ou prejuízo

In [45]:
df["lucro"] = df["valor_venda"] - df["valor_fipe"]

/tmp/ipykernel_230/3242724767.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["lucro"] = df["valor_venda"] - df["valor_fipe"]


**Resultado final**

In [46]:
df

,marca_id,modelo_id,ano_modelo,data_venda,valor_venda,taxa_servico,valor_total,km_rodados,placa,cliente,cpf,estado,status_code,valor_fipe,marca_fipe,lucro
1,59,5940,2021-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,200.0,149786.0,VW - VolksWagen,-61157.0
2,59,5940,2020-3,2025-10-01,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO ALBERTO SILVA,12345678910,RJ,200.0,141217.0,VW - VolksWagen,-52588.0
3,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA SOUZA FRANCO,98765432100,SP,200.0,25693.0,Fiat,20297.0
4,21,4828,2010-5,2025-02-15,45990.0,900.0,46890.0,98000,DEF4K56,MARIA AUGUSTA SOUZA,98765432100,SP,200.0,25693.0,Fiat,20297.0
7,23,1044,2003-1,2025-03-20,73500.0,1000.0,74500.0,65000,GHI7L89,CARLOS PEREIRA,74185296311,RJ,200.0,16875.0,GM - Chevrolet,56625.0
23,21,4828,2010-5,2025-07-10,45990.0,900.0,46890.0,98000,STU8V90,RICARDO ALVES,75315985277,RJ,200.0,25693.0,Fiat,20297.0
25,21,4828,2010-5,2025-10-07,45990.0,900.0,46890.0,98000,STU8V90,RICARDO ALVES,75315985277,XX,200.0,25693.0,Fiat,20297.0
41,59,5940,2011-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,200.0,72434.0,VW - VolksWagen,16195.0
42,59,5940,2021-3,2025-01-10,88629.0,1200.0,89829.0,120000,ABC1D23,JOAO SILVA,12345678909,SP,200.0,149786.0,VW - VolksWagen,-61157.0


Pivotamento

In [49]:
pd.pivot_table(
    df,
    values='valor_total',
    index='marca_fipe',
    columns='ano_modelo',
    aggfunc='sum',
    fill_value=0
)

ano_modelo,2003-1,2010-5,2011-3,2020-3,2021-3
marca_fipe,,,,,
Fiat,0.0,187560.0,0.0,0.0,0.0
GM - Chevrolet,74500.0,0.0,0.0,0.0,0.0
VW - VolksWagen,0.0,0.0,89829.0,89829.0,179658.0
